<a href="https://colab.research.google.com/github/Vinicius-Jose/langchain_notebooks/blob/main/RAG_EDUCATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install accelerate aiohappyeyeballs aiohttp aiosignal altair annotated-types anthropic anyio attrs backoff bm25s certifi charset-normalizer cohere colorama colorlog colpali-engine contourpy cycler datasets==3.6.0 dill distro einops fastavro filelock fonttools frozenlist fsspec gputil grpcio grpcio-tools h11 h2 hf-transfer hpack httpcore httpx huggingface-hub hyperframe idna importlib-metadata jinja2 jiter joblib jsonschema jsonschema-specifications kiwisolver markdown-it-py markupsafe matplotlib mdurl mpmath multidict multiprocess narwhals networkx numpy openai packaging pandas pdf2image peft pillow portalocker protobuf psutil pyarrow pyarrow-hotfix pydantic pydantic-core pygments pymupdf pymupdf4llm pyparsing pystemmer python-dateutil python-dotenv pytz pyyaml qdrant-client referencing regex requests requests-mock rich rich-theme-manager rpds-py safetensors scikit-learn scipy seaborn semantic-chunkers semantic-router sentence-transformers setuptools six sniffio stamina sympy tabulate tenacity threadpoolctl tiktoken tokenizers torch tqdm transformers typing-extensions tzdata urllib3 vegafusion vegafusion-python-embed vl-convert-python xxhash yarl zipp --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.7/109.7 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.9/73.9 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which ve

In [66]:
def search_online_articles_open_alex(subject:str):
    url = "https://api.openalex.org/works"
    params = {"search":subject,"filter":"open_access.oa_status:green"}
    response = requests.get(url, params=params)
    return response.json().get("results")

In [67]:
import os
import pymupdf4llm
import requests
from time import sleep
import stamina

def download_pdf_file(url:str, data_folder:str = "data"):
  if not os.path.exists(data_folder):
    os.makedirs(data_folder)
  headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': url,
        'DNT': '1',
        'Connection': 'keep-alive'
    }
  response = requests.get(url, headers=headers)
  local_pdf_path = os.path.join(data_folder, url.split('/')[-1])
  with open(local_pdf_path, 'wb') as f:
      f.write(response.content)

  md_text = pymupdf4llm.to_markdown(local_pdf_path, page_chunks=True)
  return md_text

In [68]:
from semantic_chunkers import StatisticalChunker
from semantic_router import encoders

class Chunker:
  def __init__(self,):
    self.encoder = encoders.HuggingFaceEncoder(name="sentence-transformers/all-MiniLM-L6-v2")
    self.chunker = StatisticalChunker(
    encoder=self.encoder,
    min_split_tokens=100,
    max_split_tokens=500,
    plot_chunks=False,
    enable_statistics=True,
  )
  def create_chunks(self,documents:list[str]):
    chunks = self.chunker(docs=documents)
    return chunks



In [69]:
chunker = Chunker()

def create_chunks(document: list[dict], metadata:dict = None):
  chunks = []
  for page in document:
    page_chunks = chunker.create_chunks([page["text"]])
    for chunk in page_chunks[0]:
      chunk.metadata = page["metadata"]
      if metadata:
        chunk.metadata.update(metadata)
      chunks.append(chunk)
  return chunks

In [70]:
from sentence_transformers import SentenceTransformer
import uuid
from qdrant_client import models, QdrantClient

class VectorDatabase:
  def __init__(self, collection_name:str, encoder = SentenceTransformer('all-MiniLM-L6-v2'), distance=models.Distance.COSINE ):
    self.collection_name = collection_name
    self.encoder = encoder
    self.qdrant = QdrantClient(":memory:")
    self.collection = self.qdrant.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(),
        distance=distance
    )
)

  def save_chunks(self,chunks):
    self.qdrant.upload_points(
    collection_name=self.collection_name ,
    points=[
        models.PointStruct(
            id=uuid.uuid5(uuid.NAMESPACE_URL, f"{i}-{chunk.metadata.get('title')}").hex,
            vector=self.encoder.encode([chunk.content]).tolist(),
            payload={
                "document": chunk.content,
                "metadata": chunk.metadata,
                "doc_id": i
            }
        ) for i,chunk in enumerate(chunks)

    ]
)

  def search(self,query:str, limit=2):
    hits = self.qdrant.search(
        collection_name=self.collection_name,
        query_vector=self.encoder.encode(query).tolist(),
        limit=limit
    )
    return hits

In [71]:
COLLECTION_NAME = "articles"
database = VectorDatabase(COLLECTION_NAME)

/tmp/ipython-input-2245949310.py:10: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  self.collection = self.qdrant.recreate_collection(


In [72]:
def retrieve_articles_save_chunks(subject:str):
  papers = search_online_articles_open_alex(subject)
  urls = [paper["open_access"]["oa_url"] for paper in papers[:3] ]
  documents = [download_pdf_file(url) for url in urls]
  documents_chunks = []
  for i,document in enumerate(documents):
    metadata = papers[i]
    metadata_filter = {"title":metadata.get("title"),"publication_year":metadata.get("publication_year"),"link":urls[i],
                       "author":",".join([author.get("raw_author_name") for author in papers[i].get("authorships")])}
    documents_chunks.append(create_chunks(document,metadata_filter))
    for chunks in documents_chunks:
      database.save_chunks(chunks)


In [73]:
def search_and_retrieve(subject:str):
  hits = database.search(subject,10)
  if not hits or hits[0].score < 0.3:
    print("not found")
    retrieve_articles_save_chunks(subject)
    return search_and_retrieve(subject)
  else:
    return hits

In [74]:
!pip install langchain_groq langchain langgraph langchain_core

In [75]:
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [76]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="deepseek-r1-distill-llama-70b")

In [77]:
prompt = """You are an expert AI assistant specializing in creating study materials from provided text. Your goal is to help users understand complex topics by summarizing content and generating targeted questions with answers, all while maintaining clear references to the source material.
Generate a concise study text summarizing the key information from the context. This study text must be no more than 4 paragraphs long and must include the references from the context.
Following the study text, create exactly 5 distinct questions and provide their corresponding answers. These questions and answers should be directly derived from and supported by the provided context and the study text you generated.
For each question and answer pair, you MUST include the full reference information from the context that was used to construct it. This reference information should include the author, title, source and page number if available from the context.
Ensure the questions cover different aspects of the context and are suitable for studying.
Generate the study materials STRICTLY based on the provided context and structure the output according to the specified schema. Do not attempt to perform any external actions or calls.
Given the following context, which includes information about research articles, their authors, titles, pages,source,  and content:
{context}
"""

In [78]:
from pydantic import BaseModel, Field
from typing import List

class Reference(BaseModel):
  author: str = Field(description="Author of the reference")
  title: str = Field(description="Title of the reference")
  page: str = Field(description="Page number of the reference")
  source:str = Field(description="Source of the reference")

class Question(BaseModel):
  question: str = Field(description="The question text")
  answer: str = Field(description="Answer for the question")
  reference: Reference = Field(description="Reference used to create the question")

class StudyMaterial(BaseModel):
  study_text: str  = Field(description="Summary of the context")
  questions: List[Question] = Field(description="List of questions")
  references: List[Reference] = Field(description="List of references used in Summary")

In [79]:
from langchain_core.messages import HumanMessage, SystemMessage

def create_message(subject:str):
  hits = search_and_retrieve(subject)
  context=" "
  for i,hit in enumerate(hits):
    text = hit.payload.get("document")
    metadata = hit.payload.get("metadata")
    context += f"{i}. Author {metadata.get('author')} - Title: {metadata.get('title')} - Page {metadata.get('page')} - Source:{metadata.get('link')} - Content: {text} \n"
  messages = [SystemMessage(content=prompt.format(context=context)),HumanMessage(content=subject)]
  return messages


In [80]:
llm_output = llm.with_structured_output(StudyMaterial)


In [81]:
messages = create_message("Langchain and Langgraph")

/tmp/ipython-input-2245949310.py:36: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = self.qdrant.search(


not found


2025-08-28 17:25:19 INFO semantic_chunkers.utils.logger Single document exceeds the maximum token limit of 500. Splitting to sentences before semantically merging.


  0%|          | 0/2 [00:00<?, ?it/s]

Chunking Statistics:
  - Total Documents: 64
  - Total Chunks: 3
  - Chunks by Threshold: 2
  - Chunks by Max Chunk Size: 0
  - Last Chunk: 1
  - Minimum Token Size of Chunk: 133
  - Maximum Token Size of Chunk: 237
  - Similarity Chunk Ratio: 0.67


...

  0%|          | 0/1 [00:00<?, ?it/s]

2025-08-28 17:25:48 INFO semantic_chunkers.utils.logger Single document exceeds the maximum token limit of 500. Splitting to sentences before semantically merging.


Chunking Statistics:
  - Total Documents: 58
  - Total Chunks: 3
  - Chunks by Threshold: 2
  - Chunks by Max Chunk Size: 0
  - Last Chunk: 1
  - Minimum Token Size of Chunk: 110
  - Maximum Token Size of Chunk: 293
  - Similarity Chunk Ratio: 0.67


  0%|          | 0/2 [00:00<?, ?it/s]

Chunking Statistics:
  - Total Documents: 64
  - Total Chunks: 5
  - Chunks by Threshold: 4
  - Chunks by Max Chunk Size: 0
  - Last Chunk: 1
  - Minimum Token Size of Chunk: 108
  - Maximum Token Size of Chunk: 142
  - Similarity Chunk Ratio: 0.80
Chunking Statistics:
  - Total Documents: 18
  - Total Chunks: 1
  - Chunks by Threshold: 0
  - Chunks by Max Chunk Size: 0
  - Last Chunk: 1
  - Minimum Token Size of Chunk: 190
  - Maximum Token Size of Chunk: 190
  - Similarity Chunk Ratio: 0.00


In [82]:
response = llm_output.invoke(messages)

In [83]:
response.study_text

'LangGraph is a framework built on LangChain that simplifies the creation and management of agents for machine translation tasks. It allows agents to maintain dialogue context and handle translation tasks coherently. The framework supports multilingual translations, enabling agents to translate between languages like English, French, and Japanese. LangGraph-based agents leverage large language models to improve translation accuracy and maintain context. Experimental results demonstrate the effectiveness of LangGraph in enhancing machine translation tasks, showing good translation effects with 75,000 words used for training.'

In [84]:
response.questions

[Question(question='What is LangGraph and what does it simplify in the context of machine translation?', answer='LangGraph is a framework built on LangChain that simplifies the creation and management of agents for machine translation tasks.', reference=Reference(author='Wang, Jialin,Duan, Zhihua', title='Agent AI with LangGraph: A Modular Framework for Enhancing Machine Translation Using Large Language Models', page='6', source='http://arxiv.org/pdf/2412.03801')),
 Question(question='What is the role of agents in LangGraph-based machine translation?', answer='Agents in LangGraph-based machine translation are responsible for translating text between specific languages, such as English, French, and Japanese, by leveraging large language models.', reference=Reference(author='Wang, Jialin,Duan, Zhihua', title='Agent AI with LangGraph: A Modular Framework for Enhancing Machine Translation Using Large Language Models', page='8', source='http://arxiv.org/pdf/2412.03801')),
 Question(question

In [85]:
response.references

[Reference(author='Wang, Jialin,Duan, Zhihua', title='Agent AI with LangGraph: A Modular Framework for Enhancing Machine Translation Using Large Language Models', page='6', source='http://arxiv.org/pdf/2412.03801'),
 Reference(author='Wang, Jialin,Duan, Zhihua', title='Agent AI with LangGraph: A Modular Framework for Enhancing Machine Translation Using Large Language Models', page='8', source='http://arxiv.org/pdf/2412.03801'),
 Reference(author='Wang, Jialin,Duan, Zhihua', title='Agent AI with LangGraph: A Modular Framework for Enhancing Machine Translation Using Large Language Models', page='11', source='http://arxiv.org/pdf/2412.03801'),
 Reference(author='Wang, Jialin,Duan, Zhihua', title='Agent AI with LangGraph: A Modular Framework for Enhancing Machine Translation Using Large Language Models', page='8', source='http://arxiv.org/pdf/2412.03801'),
 Reference(author='Wang, Jialin,Duan, Zhihua', title='Agent AI with LangGraph: A Modular Framework for Enhancing Machine Translation Us